# Staggered DiD with Multiple Treatment Cohorts

**Module 04 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Generate and explore a staggered adoption panel dataset
2. Show that standard TWFE produces biased estimates under heterogeneous treatment effects
3. Implement the Goodman-Bacon decomposition to diagnose TWFE bias
4. Estimate $ATT(g,t)$ using the Callaway-Sant'Anna approach manually
5. Compare TWFE and cohort-robust estimates on the same dataset

## Background

State-level policy adoptions — anti-smoking laws, right-to-carry gun laws, healthcare expansions — rarely happen simultaneously. States adopt in waves over many years. This **staggered adoption** setting is where the TWFE estimator can mislead, as shown by Goodman-Bacon (2021), Callaway & Sant'Anna (2021), and Sun & Abraham (2021).

We'll use a simulated job training program that states adopt at different times, with treatment effects that grow over time (dynamic effects).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2021)  # Callaway & Sant'Anna year
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Generate Staggered Adoption Data

We simulate 50 states over 20 periods (years). States adopt a job training program in one of three cohorts:
- **Cohort 2004** (15 states): treated starting period 5
- **Cohort 2008** (15 states): treated starting period 9
- **Cohort 2012** (10 states): treated starting period 13
- **Never treated** (10 states): never adopt the program

True treatment effect: **dynamic and growing** — the program takes time to build

$$\tau_{it} = 2 + 0.4 \times (t - G_i) \quad \text{for } t \geq G_i$$

In [ ]:
# Parameters
N_PERIODS = 20
PERIODS = list(range(1, N_PERIODS + 1))
BASE_EFFECT = 2.0
GROWTH_EFFECT = 0.4

# Cohort structure: (n_states, first_treatment_period)
cohort_config = [
    (15, 5),   # Early cohort
    (15, 9),   # Mid cohort
    (10, 13),  # Late cohort
    (10, np.inf),  # Never treated
]

# Generate panel
rows = []
state_id = 0
for n_states, g in cohort_config:
    for _ in range(n_states):
        unit_fe = np.random.normal(0, 2)   # unit-specific level
        unit_trend = np.random.normal(0.3, 0.1)  # unit-specific time trend
        for t in PERIODS:
            # Determine treatment status
            treated = 1 if (t >= g and g != np.inf) else 0
            event_time = (t - g) if g != np.inf else np.nan
            
            # True treatment effect (grows with event time)
            if treated:
                true_tau = BASE_EFFECT + GROWTH_EFFECT * (t - g)
            else:
                true_tau = 0
            
            # Outcome: common time trend + unit FE + individual trend + treatment + noise
            y = (5.0
                 + 0.5 * t
                 + unit_fe
                 + unit_trend * t
                 + true_tau
                 + np.random.normal(0, 1.5))
            
            rows.append({
                'state': state_id,
                'period': t,
                'cohort': g,
                'treated': treated,
                'event_time': event_time,
                'true_tau': true_tau,
                'outcome': y,
            })
        state_id += 1

df = pd.DataFrame(rows)

# Create dummy variable: ever treated
df['ever_treated'] = (df['cohort'] != np.inf).astype(int)
df['cohort_label'] = df['cohort'].apply(
    lambda g: f'Cohort {int(g)}' if g != np.inf else 'Never treated'
)

print(f"Panel shape: {df.shape}")
print(f"\nStates per cohort:")
print(df.groupby('cohort_label')['state'].nunique())
print(f"\nTrue ATT range: [{df[df['treated']==1]['true_tau'].min():.1f}, {df[df['treated']==1]['true_tau'].max():.1f}]")

## 2. Visualise the Raw Data

Plot the average outcome by cohort over time. Look for: (1) parallel pre-trends across cohorts, (2) different post-treatment trajectories due to the dynamic effects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    'Cohort 5': 'steelblue',
    'Cohort 9': 'darkorange',
    'Cohort 13': 'seagreen',
    'Never treated': 'gray'
}
treatment_periods = {5: 5, 9: 9, 13: 13}

# Left: raw outcome means
ax = axes[0]
for cohort_label in df['cohort_label'].unique():
    subset = df[df['cohort_label'] == cohort_label].groupby('period')['outcome'].mean()
    short_label = cohort_label.replace('Cohort ', 'g=')
    ax.plot(subset.index, subset.values, label=short_label,
            color=colors.get(cohort_label, 'purple'), linewidth=2)

# Add treatment lines
for period in [5, 9, 13]:
    ax.axvline(period, color='red', linestyle=':', alpha=0.4, linewidth=1)

ax.set_xlabel('Period')
ax.set_ylabel('Mean Outcome')
ax.set_title('Mean Outcome by Cohort Over Time\n(dashed lines = treatment adoption)')
ax.legend()

# Right: de-meaned to isolate treatment effects
ax2 = axes[1]
for cohort_label, g in [('Cohort 5', 5), ('Cohort 9', 9), ('Cohort 13', 13)]:
    subset = df[df['cohort_label'] == cohort_label].groupby('period')['true_tau'].mean()
    short_label = cohort_label.replace('Cohort ', 'g=')
    ax2.plot(subset.index - g, subset.values,
             label=short_label, color=colors[cohort_label], linewidth=2, marker='o', markersize=4)

ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Treatment onset')
ax2.axhline(0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('Event time (periods since treatment)')
ax2.set_ylabel('True ATT(g, t)')
ax2.set_title('True Treatment Effect by Cohort\n(growing dynamic effects)')
ax2.legend()

plt.tight_layout()
plt.show()

## 3. TWFE Estimator — The Biased Benchmark

Standard Two-Way Fixed Effects regression with the binary treatment indicator:

In [ ]:
# Two-way fixed effects regression
model_twfe = smf.ols('outcome ~ treated + C(state) + C(period)', data=df).fit(cov_type='HC1')
twfe_estimate = model_twfe.params['treated']
twfe_ci = model_twfe.conf_int().loc['treated'].values

# True overall ATT: weighted average of true effects for all treated observations
true_att = df[df['treated'] == 1]['true_tau'].mean()

print("=" * 50)
print("TWFE ESTIMATE vs TRUE ATT")
print("=" * 50)
print(f"TWFE estimate:     {twfe_estimate:.3f} [95% CI: {twfe_ci[0]:.2f}, {twfe_ci[1]:.2f}]")
print(f"True overall ATT:  {true_att:.3f}")
print(f"Bias:              {twfe_estimate - true_att:+.3f}")
print()
print("The TWFE estimate is biased because early-treated states serve")
print("as controls for late-treated states. With growing treatment effects,")
print("this creates downward bias.")

## 4. Manual Callaway-Sant'Anna: Group-Time ATTs

The CS approach estimates $ATT(g,t)$ for each cohort $g$ and period $t$, using only never-treated states as controls. Then aggregates.

In [ ]:
def estimate_att_gt(df, g, t, control_group='never_treated'):
    """
    Estimate ATT(g, t): the treatment effect for cohort g at period t.
    Uses a 2x2 DiD comparing cohort g at periods t vs g-1,
    against never-treated units.
    """
    # Treated cohort: states first treated at period g
    treated_g = df[df['cohort'] == g]
    
    if control_group == 'never_treated':
        control = df[df['cohort'] == np.inf]
    else:  # not yet treated
        control = df[(df['cohort'] > t) | (df['cohort'] == np.inf)]

    # We compare periods g-1 (baseline) and t
    baseline_period = g - 1
    
    treated_t = treated_g[treated_g['period'] == t]['outcome'].mean()
    treated_base = treated_g[treated_g['period'] == baseline_period]['outcome'].mean()
    control_t = control[control['period'] == t]['outcome'].mean()
    control_base = control[control['period'] == baseline_period]['outcome'].mean()
    
    if any(pd.isna([treated_t, treated_base, control_t, control_base])):
        return np.nan
    
    att_gt = (treated_t - treated_base) - (control_t - control_base)
    return att_gt


# Compute ATT(g, t) for all cohorts and periods
cohorts = [5, 9, 13]
att_results = []

for g in cohorts:
    for t in PERIODS:
        if t == g - 1:  # baseline period, skip
            continue
        att_val = estimate_att_gt(df, g, t)
        true_val = df[(df['cohort'] == g) & (df['period'] == t)]['true_tau'].mean()
        att_results.append({
            'cohort': g, 'period': t,
            'event_time': t - g,
            'att_gt': att_val,
            'true_att_gt': true_val
        })

att_df = pd.DataFrame(att_results)
print(f"Computed {len(att_df)} group-time ATT estimates")
print("\nSample of ATT(g,t) estimates:")
print(att_df[att_df['event_time'].isin([0, 2, 4])].round(2).to_string(index=False))

In [ ]:
# Aggregate to event study: average ATT(g, g+k) across cohorts
event_study = att_df.groupby('event_time').agg(
    att_mean=('att_gt', 'mean'),
    true_mean=('true_att_gt', 'mean'),
    n=('att_gt', 'count')
).reset_index()

# Filter to reasonable event time range
event_study = event_study[event_study['event_time'].between(-4, 7)].copy()

# Overall ATT: average across all post-treatment observations
post_att = att_df[att_df['event_time'] >= 0]
cs_overall_att = post_att['att_gt'].mean()
true_overall_att = df[df['treated'] == 1]['true_tau'].mean()

print("Callaway-Sant'Anna vs TWFE:")
print(f"  CS overall ATT:  {cs_overall_att:.3f}")
print(f"  TWFE estimate:   {twfe_estimate:.3f}")
print(f"  True ATT:        {true_overall_att:.3f}")

In [ ]:
# Plot event study: estimated vs true
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: event study
ax = axes[0]
ax.plot(event_study['event_time'], event_study['att_mean'],
        'o-', color='steelblue', linewidth=2, markersize=6, label='CS Estimate')
ax.plot(event_study['event_time'], event_study['true_mean'],
        's--', color='green', linewidth=2, markersize=6, label='True ATT', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='Treatment onset')
ax.axhline(0, color='black', linestyle='-', alpha=0.2)

ax.set_xlabel('Event Time (periods since treatment)')
ax.set_ylabel('ATT Estimate')
ax.set_title("Event Study: CS Estimates vs True ATT\n(aggregated across cohorts)")
ax.legend()

# Right: comparison of methods
ax2 = axes[1]
estimates = [
    ('True ATT', true_overall_att, None),
    ('TWFE', twfe_estimate, twfe_ci),
    ('CS (manual)', cs_overall_att, None),
]

for i, (name, est, ci) in enumerate(estimates):
    color = 'green' if 'True' in name else ('red' if 'TWFE' in name else 'steelblue')
    ax2.scatter(est, i, color=color, s=100, zorder=5)
    if ci is not None:
        ax2.plot(ci, [i, i], color=color, linewidth=2.5)
    ax2.text(est + 0.1, i + 0.1, f'{est:.2f}', fontsize=10)

ax2.axvline(true_overall_att, color='green', linestyle=':', alpha=0.5)
ax2.set_yticks(range(len(estimates)))
ax2.set_yticklabels([e[0] for e in estimates])
ax2.set_xlabel('ATT Estimate')
ax2.set_title('Overall ATT: TWFE vs CS\n(TWFE is biased; CS recovers true effect)')
ax2.set_xlim(0, true_overall_att + 2)

plt.tight_layout()
plt.show()

## 5. Cohort-Specific Effects

Break down the ATT by cohort — earlier cohorts accumulate more treatment time and should show larger average effects.

In [ ]:
# Cohort-specific ATTs
post_att_df = att_df[att_df['event_time'] >= 0]

print("Cohort-Specific Average Treatment Effects")
print("=" * 55)
print(f"{'Cohort':<12} {'CS Estimate':>12} {'True ATT':>12} {'Bias':>8}")
print("-" * 55)
for g in cohorts:
    subset = post_att_df[post_att_df['cohort'] == g]
    cs_att = subset['att_gt'].mean()
    true_att_g = subset['true_att_gt'].mean()
    print(f"g={g:<10} {cs_att:>12.3f} {true_att_g:>12.3f} {cs_att - true_att_g:>+8.3f}")
print("-" * 55)
overall_cs = post_att_df['att_gt'].mean()
overall_true = df[df['treated']==1]['true_tau'].mean()
print(f"{'Overall':<12} {overall_cs:>12.3f} {overall_true:>12.3f} {overall_cs - overall_true:>+8.3f}")

print()
print("The CS estimator recovers close-to-true ATTs for each cohort.")
print("Earlier cohorts show larger average effects because they accumulated")
print("more treatment periods (where effects are larger due to dynamic growth).")

## 6. Pre-Trend Test

The event study pre-period coefficients (event_time < 0) should be near zero if parallel trends holds.

In [ ]:
# Pre-trend coefficients
pre_trend = event_study[event_study['event_time'] < 0]

print("Pre-Period Event Study Coefficients (should be near 0):")
print(pre_trend[['event_time', 'att_mean', 'true_mean']].round(3).to_string(index=False))

# Simple test: are pre-period estimates jointly close to zero?
pre_vals = pre_trend['att_mean'].values
print(f"\nMean absolute pre-trend deviation: {np.abs(pre_vals).mean():.3f}")
print(f"Max absolute pre-trend deviation: {np.abs(pre_vals).max():.3f}")

# Visual: zoom into pre-period
fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(pre_trend['event_time'], pre_trend['att_mean'],
           color='steelblue', s=80, zorder=5, label='CS Estimate (pre-period)')
ax.axhline(0, color='black', linestyle='-', alpha=0.3)
ax.axvline(0, color='red', linestyle='--', alpha=0.6, label='Treatment onset')
ax.fill_between([-5, 0], -1, 1, alpha=0.05, color='green',
                 label='±1 unit band')
ax.set_xlabel('Event Time')
ax.set_ylabel('ATT Estimate')
ax.set_title('Pre-Trend Check: Event Study Coefficients (k < 0)\nShould cluster around zero')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Comparison Summary

The key takeaway: when treatment effects are heterogeneous and growing over time, TWFE underestimates the true ATT because early-treated units serve as biased controls for late-treated units.

In [ ]:
print("=" * 60)
print("STAGGERED DiD: SUMMARY OF FINDINGS")
print("=" * 60)
print(f"{'Estimator':<25} {'Estimate':>10} {'Bias':>10}")
print("-" * 60)
print(f"{'True ATT':<25} {true_overall_att:>10.3f} {'—':>10}")
print(f"{'TWFE (biased)':<25} {twfe_estimate:>10.3f} {twfe_estimate - true_overall_att:>+10.3f}")
print(f"{'CS (cohort-robust)':<25} {cs_overall_att:>10.3f} {cs_overall_att - true_overall_att:>+10.3f}")
print("=" * 60)
print()
print("Key insight: TWFE downward bias arises because:")
print("  1. Early-treated cohorts are used as controls for later cohorts")
print("  2. Those early-treated controls already have treatment effects")
print("     embedded in their outcomes")
print("  3. With growing effects, this contamination is especially severe")
print()
print("Callaway-Sant'Anna uses only never-treated units as controls,")
print("avoiding this contamination entirely.")

## Self-Check

1. Set `GROWTH_EFFECT = 0` so treatment effects are constant at `BASE_EFFECT = 2.0` for all cohorts and all event times. Rerun the notebook. Does TWFE now recover the true ATT? (Under homogeneous effects, TWFE is unbiased.)

2. Change the never-treated group to a very small sample (2 states). How does this affect the CS estimates? What does this suggest about the importance of having a large, representative control group?

3. Add a violation of parallel trends: for cohort 5, add a pre-existing upward trend of +0.5 per period in the pre-period. Does this show up in the event study pre-trend plot?

4. Implement the Sun-Abraham approach: create binary indicators for each cohort × event-time combination and run OLS. Compare the resulting event study to the CS estimates above.

---
**Next:** `03_event_study_plots.ipynb` — Event study visualisation and pre-trend testing in depth